# MCD-rPPG Dataset: Preprocessing, Chunking & ROI Extraction Showcase## OverviewThis notebook demonstrates the complete preprocessing pipeline for the MCD-rPPG dataset:1. **Video Frame Analysis**: Verify frame counts and alignment with PPG sync files2. **Chunking Strategy**: Split videos into 450-frame chunks with 150-frame overlap3. **ROI Extraction**: Extract 9 facial regions using MediaPipe landmarks4. **Data Saving**: Save processed chunks as NPZ files with ROIs, PPG signals, and vital signs5. **Visualization**: Showcase results for different camera orientations### Key Parameters:- **Chunk Size**: 450 frames- **Overlap**: 150 frames (last 150 of chunk N = first 150 of chunk N+1)- **ROIs**: 9 regions (full_face, forehead, left_eye, right_eye, nose, mouth, chin, left_iris, right_iris)- **Output**: NPZ files with ROI data, PPG signals (value + time deltas), and vital signs### Jitter Reduction:MediaPipe landmarks are smoothed using a moving average window to prevent frame-to-frame jitter in ROI masks.

In [ ]:
# Install required packages!pip install imageio[ffmpeg] -q!pip install mediapipe -q!pip install scikit-video -q!pip install opencv-python -q!pip install numba -q!pip install scipy -q

In [ ]:
import osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport matplotlib.patches as patchesimport seaborn as snsimport imageioimport cv2from IPython.display import displayfrom tqdm import tqdmimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8')# ConfigurationDATASET_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/'DB_PATH = os.path.join(DATASET_PATH, 'db.csv')MEDIAPIPE_MODEL_PATH = '/home/cristic/face_landmarker.task'OUTPUT_PATH = '/home/cristic/preprocessed_data'# Chunking parametersCHUNK_SIZE = 450OVERLAP_SIZE = 150# ROI ConfigurationROIS = {    'full_face': list(range(468)),    'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],    'left_eye': list(range(22, 53)),    'right_eye': list(range(252, 283)),    'nose': list(range(1, 21)) + list(range(195, 221)),    'mouth': list(range(60, 81)) + list(range(290, 321)),    'chin': list(range(150, 171)) + list(range(370, 391)),    'left_iris': list(range(468, 473)),    'right_iris': list(range(473, 478))}ROI_BOX_SIZE = (24, 24)SMOOTHING_WINDOW = 5print('Configuration loaded successfully')

## 1. Load Database and Prepare Metadata

In [ ]:
df = pd.read_csv(DB_PATH)meta_df = df.copy()file_columns = ['ecg', 'video', 'meta', 'ppg_sync']for col in file_columns:    if col in meta_df.columns:        meta_df[f'{col}_full'] = meta_df[col].apply(            lambda x: os.path.join(DATASET_PATH, x) if not os.path.isabs(x) else x        )meta_df['subject_id'] = meta_df['patient_id']meta_df['condition'] = meta_df['step']meta_df['camera_type'] = meta_df['camera']meta_df['view_type'] = meta_df['view']print(f'Total videos: {len(meta_df)}')print(f'Camera types: {meta_df["camera_type"].unique()}')

## 2. Video Frame Analysis and PPG Sync Verification

In [ ]:
def count_video_frames_ffmpeg(video_path):    try:        reader = imageio.get_reader(video_path, 'ffmpeg')        n_frames = reader.count_frames()        reader.close()        return n_frames    except Exception as e:        print(f'Error: {e}')        return 0def count_ppg_sync_rows(ppg_sync_path):    try:        if ppg_sync_path.endswith('.npy'):            data = np.load(ppg_sync_path)        elif ppg_sync_path.endswith('.txt'):            data = np.loadtxt(ppg_sync_path)        else:            data = np.load(ppg_sync_path)        return len(data)    except Exception as e:        print(f'Error: {e}')        return 0sample_videos = meta_df.dropna(subset=['video_full', 'ppg_sync_full']).head(5)for idx, row in sample_videos.iterrows():    n_frames = count_video_frames_ffmpeg(row['video_full'])    n_ppg = count_ppg_sync_rows(row['ppg_sync_full'])    match = 'MATCH' if n_frames == n_ppg else 'MISMATCH'    print(f'{row["patient_id"]}: video={n_frames}, ppg={n_ppg} [{match}]')

## 3. MediaPipe Landmark Detection with Temporal Smoothing

In [ ]:
try:    import mediapipe as mp    from mediapipe.tasks import python    from mediapipe.tasks.python import vision    MEDIAPIPE_AVAILABLE = Trueexcept ImportError:    MEDIAPIPE_AVAILABLE = False    print('MediaPipe not available')class TemporalSmoother:    def __init__(self, window_size=5):        self.window_size = window_size        self.history = []    def smooth(self, landmarks):        self.history.append(landmarks.copy())        if len(self.history) < self.window_size:            return landmarks        weights = [(self.window_size - i) / self.window_size for i in range(len(self.history))]        smoothed = np.zeros_like(landmarks)        for i, w in enumerate(weights):            smoothed += self.history[i] * w        smoothed /= sum(weights)        if len(self.history) > self.window_size * 2:            self.history.pop(0)        return smoothedclass MediaPipeLandmarkDetector:    def __init__(self, model_path, smoothing_window=5):        self.model_path = model_path        self.smoothing_window = smoothing_window        self.smoother = TemporalSmoother(smoothing_window)        self.detector = None        self.frame_count = 0        self.initialize_detector()    def initialize_detector(self):        if not MEDIAPIPE_AVAILABLE:            raise RuntimeError('MediaPipe not available')        base_options = python.BaseOptions(model_asset_path=self.model_path)        options = vision.FaceLandmarkerOptions(            base_options=base_options,            output_face_blendshapes=True,            output_facial_transformation_matrixes=True,            num_faces=1        )        self.detector = vision.FaceLandmarker.create_from_options(options)    def detect_landmarks(self, frame):        if frame.dtype != np.uint8:            frame = (frame * 255).astype(np.uint8)        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)        try:            result = self.detector.detect_for_video(mp_image, self.frame_count)            self.frame_count += 1            if result and result.face_landmarks:                face_landmarks = result.face_landmarks[0]                h, w = frame.shape[:2]                points = np.array([(lm.x * w, lm.y * h) for lm in face_landmarks], dtype='float32')                return self.smoother.smooth(points)            return None        except Exception as e:            print(f'Error: {e}')            return None    def reset(self):        self.frame_count = 0        self.smoother.history = []if MEDIAPIPE_AVAILABLE:    detector = MediaPipeLandmarkDetector(MEDIAPIPE_MODEL_PATH, smoothing_window=SMOOTHING_WINDOW)    print('MediaPipe detector initialized with temporal smoothing')

## 4. Chunking Strategy Implementation

In [ ]:
def create_chunks(n_frames, chunk_size=450, overlap_size=150):    chunks = []    chunk_idx = 0    start = 0    while start < n_frames:        end = min(start + chunk_size, n_frames)        chunks.append((start, end, chunk_idx))        if end == n_frames:            break        start = end - overlap_size        chunk_idx += 1    return chunkstest_frames = 1800chunks = create_chunks(test_frames, CHUNK_SIZE, OVERLAP_SIZE)print(f'Video with {test_frames} frames -> {len(chunks)} chunks')for start, end, idx in chunks:    print(f'  Chunk {idx}: frames {start}-{end} ({end-start} frames)')

## 5. ROI Extraction with Fixed Bounding Boxes

In [ ]:
def extract_roi_bbox(landmarks, roi_indices, frame_shape, box_size=(24, 24)):    valid_indices = [i for i in roi_indices if i < len(landmarks)]    if not valid_indices:        return (0, 0, box_size[0], box_size[1])    roi_points = landmarks[valid_indices]    center_x = int(np.mean(roi_points[:, 0]))    center_y = int(np.mean(roi_points[:, 1]))    box_w, box_h = box_size    x = max(0, center_x - box_w // 2)    y = max(0, center_y - box_h // 2)    x = min(x, frame_shape[1] - box_w)    y = min(y, frame_shape[0] - box_h)    return (x, y, box_w, box_h)def extract_roi_region(frame, bbox):    x, y, w, h = bbox    return frame[y:y+h, x:x+w]def extract_all_rois_for_frame(frame, landmarks, rois_dict, box_size=(24, 24)):    frame_shape = frame.shape[:2]    roi_data = {}    for roi_name, roi_indices in rois_dict.items():        bbox = extract_roi_bbox(landmarks, roi_indices, frame_shape, box_size)        roi_region = extract_roi_region(frame, bbox)        roi_data[roi_name] = roi_region    return roi_dataprint('ROI extraction functions ready')

## 6. Complete Preprocessing Pipeline

In [ ]:
def load_ppg_sync_data(ppg_sync_path):    try:        if ppg_sync_path.endswith('.npy'):            data = np.load(ppg_sync_path)        elif ppg_sync_path.endswith('.txt'):            data = np.loadtxt(ppg_sync_path)        else:            data = np.load(ppg_sync_path)        if data.ndim == 1:            data = data.reshape(-1, 1)        return data    except Exception as e:        print(f'Error: {e}')        return np.array([])def load_video_frames(video_path, start_frame=0, end_frame=None):    try:        reader = imageio.get_reader(video_path, 'ffmpeg')        total_frames = reader.count_frames()        if end_frame is None:            end_frame = total_frames        frames = []        for i in range(start_frame, end_frame):            frames.append(reader.get_next_data())        reader.close()        return np.array(frames)    except Exception as e:        print(f'Error: {e}')        return np.array([])def process_video_chunk(video_path, ppg_sync_path, meta_data, chunk_start, chunk_end, chunk_idx):    video_frames = load_video_frames(video_path, chunk_start, chunk_end)    if len(video_frames) == 0:        return None    ppg_sync_data = load_ppg_sync_data(ppg_sync_path)    if len(ppg_sync_data) == 0:        return None    chunk_ppg = ppg_sync_data[chunk_start:chunk_end]    if chunk_ppg.shape[1] >= 2:        ppg_values = chunk_ppg[:, 0]        time_deltas = chunk_ppg[:, 1]    else:        ppg_values = chunk_ppg[:, 0] if chunk_ppg.ndim > 1 else chunk_ppg        time_deltas = np.zeros_like(ppg_values)    detector.reset()    chunk_landmarks = []    for frame in video_frames:        lms = detector.detect_landmarks(frame)        if lms is not None:            chunk_landmarks.append(lms)        elif chunk_landmarks:            chunk_landmarks.append(chunk_landmarks[-1].copy())        else:            return None    chunk_landmarks = np.array(chunk_landmarks)    roi_data = {}    for roi_name, roi_indices in ROIS.items():        roi_frames = []        for frame_idx, frame in enumerate(video_frames):            landmarks = chunk_landmarks[frame_idx]            bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)            roi_region = extract_roi_region(frame, bbox)            roi_frames.append(roi_region)        roi_data[roi_name] = np.array(roi_frames)    chunk_meta = {        'subject_id': meta_data.get('subject_id'),        'condition': meta_data.get('condition'),        'camera_type': meta_data.get('camera_type'),        'view_type': meta_data.get('view_type'),        'chunk_index': chunk_idx,        'start_frame': chunk_start,        'end_frame': chunk_end,        'num_frames': chunk_end - chunk_start    }    vital_signs = {}    vital_cols = ['weight', 'height', 'bmi', 'age', 'upper_ap', 'lower_ap',                  'saturation', 'temperature', 'hemoglobin', 'glycated_hemoglobin',                  'cholesterol', 'respiratory', 'rigidity', 'pulse', 'stress']    for col in vital_cols:        if col in meta_data:            vital_signs[col] = meta_data[col]    return {        'roi_data': roi_data,        'ppg_values': ppg_values,        'time_deltas': time_deltas,        'landmarks': chunk_landmarks,        'metadata': chunk_meta,        'vital_signs': vital_signs    }def save_chunk_as_npz(chunk_data, output_path):    try:        os.makedirs(output_path, exist_ok=True)        save_data = {}        for roi_name, roi_frames in chunk_data['roi_data'].items():            save_data[f'roi_{roi_name}'] = roi_frames        save_data['ppg_values'] = chunk_data['ppg_values']        save_data['time_deltas'] = chunk_data['time_deltas']        save_data['landmarks'] = chunk_data['landmarks']        for key, value in chunk_data['metadata'].items():            save_data[f'meta_{key}'] = value        for key, value in chunk_data['vital_signs'].items():            save_data[f'vital_{key}'] = value        subject_id = chunk_data['metadata']['subject_id']        camera = chunk_data['metadata']['camera_type']        condition = chunk_data['metadata']['condition']        chunk_idx = chunk_data['metadata']['chunk_index']        filename = f'{subject_id}_{camera}_{condition}_chunk{chunk_idx}.npz'        filepath = os.path.join(output_path, filename)        np.savez_compressed(filepath, **save_data)        print(f'  Saved: {filepath}')        return filepath    except Exception as e:        print(f'Error: {e}')        return Nonedef process_complete_video(video_row, output_path=OUTPUT_PATH):    video_path = video_row['video_full']    ppg_sync_path = video_row['ppg_sync_full']    print(f'Processing: {os.path.basename(video_path)}')    n_frames = count_video_frames_ffmpeg(video_path)    chunks = create_chunks(n_frames, CHUNK_SIZE, OVERLAP_SIZE)    meta_data = {        'subject_id': video_row['patient_id'],        'condition': video_row['condition'],        'camera_type': video_row['camera_type'],        'view_type': video_row['view_type']    }    vital_cols = ['weight', 'height', 'bmi', 'age', 'upper_ap', 'lower_ap',                  'saturation', 'temperature', 'hemoglobin', 'glycated_hemoglobin',                  'cholesterol', 'respiratory', 'rigidity', 'pulse', 'stress']    for col in vital_cols:        if col in video_row:            meta_data[col] = video_row[col]    saved_files = []    for start, end, chunk_idx in tqdm(chunks, desc='Processing chunks'):        chunk_data = process_video_chunk(video_path, ppg_sync_path, meta_data, start, end, chunk_idx)        if chunk_data is not None:            filepath = save_chunk_as_npz(chunk_data, output_path)            if filepath:                saved_files.append(filepath)    return saved_filesprint('Preprocessing pipeline functions ready')

## 7. Showcase: Process and Visualize One Video from Each Camera Orientation

In [ ]:
if MEDIAPIPE_AVAILABLE:    print('SHOWCASE: Processing one video from each camera orientation')    camera_types = meta_df['camera_type'].unique()    showcase_videos = []    for camera in camera_types:        samples = meta_df[meta_df['camera_type'] == camera].dropna(subset=['video_full', 'ppg_sync_full'])        if len(samples) > 0:            showcase_videos.append(samples.iloc[0])    showcase_results = []    for video_row in showcase_videos:        video_path = video_row['video_full']        ppg_sync_path = video_row['ppg_sync_full']        n_frames = count_video_frames_ffmpeg(video_path)        chunks = create_chunks(n_frames, CHUNK_SIZE, OVERLAP_SIZE)        if chunks:            start, end, chunk_idx = chunks[0]            meta_data = {                'subject_id': video_row['patient_id'],                'condition': video_row['condition'],                'camera_type': video_row['camera_type'],                'view_type': video_row['view_type']            }            vital_cols = ['pulse', 'saturation', 'upper_ap', 'lower_ap']            for col in vital_cols:                if col in video_row:                    meta_data[col] = video_row[col]            chunk_data = process_video_chunk(video_path, ppg_sync_path, meta_data, start, end, chunk_idx)            if chunk_data is not None:                showcase_results.append(chunk_data)                print(f'  Processed {video_row["camera_type"]} camera')    print(f'Successfully processed {len(showcase_results)} camera orientations')

In [ ]:
if MEDIAPIPE_AVAILABLE and 'showcase_results' in locals() and showcase_results:    for chunk_data in showcase_results:        camera = chunk_data['metadata']['camera_type']        subject = chunk_data['metadata']['subject_id']        condition = chunk_data['metadata']['condition']        n_frames = chunk_data['metadata']['num_frames']        start_frame = chunk_data['metadata']['start_frame']        end_frame = chunk_data['metadata']['end_frame']        print(f'--- {camera} Camera (Subject {subject}, {condition}) ---')        print(f'Chunk: frames {start_frame}-{end_frame} ({n_frames} frames)')        vital_signs = chunk_data['vital_signs']        if vital_signs:            print('Vital Signs:')            for key, value in vital_signs.items():                print(f'  {key}: {value}')        video_frames = chunk_data['roi_data']['full_face']        frame_indices = [0, min(n_frames // 2, n_frames - 1)]        fig, axes = plt.subplots(2, len(ROIS), figsize=(20, 8))        for row_idx, frame_idx in enumerate(frame_indices):            for col_idx, (roi_name, roi_frames) in enumerate(chunk_data['roi_data'].items()):                axes[row_idx, col_idx].imshow(roi_frames[frame_idx])                title = roi_name if row_idx == 0 else ''                axes[row_idx, col_idx].set_title(title)                axes[row_idx, col_idx].axis('off')        plt.suptitle(f'{camera} Camera - ROI Frames')        plt.tight_layout()        plt.show()        ppg_values = chunk_data['ppg_values']        time_deltas = chunk_data['time_deltas']        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))        ax1.plot(ppg_values, color='red', linewidth=1.5)        ax1.set_title(f'PPG Signal - {camera} Camera')        ax1.set_xlabel('Frame Index')        ax1.set_ylabel('PPG Value')        ax1.grid(True, alpha=0.3)        ax2.plot(time_deltas, color='blue', linewidth=1.5)        ax2.set_title('Time Deltas')        ax2.set_xlabel('Frame Index')        ax2.set_ylabel('Time Delta (seconds)')        ax2.grid(True, alpha=0.3)        plt.tight_layout()        plt.show()        save_path = os.path.join(OUTPUT_PATH, 'showcase')        os.makedirs(save_path, exist_ok=True)        save_chunk_as_npz(chunk_data, save_path)    print('SHOWCASE COMPLETE')

## 8. Inference Function for Real Videos

In [ ]:
def chunk_video_for_inference(video_path, chunk_size=450, overlap_size=150):    n_frames = count_video_frames_ffmpeg(video_path)    chunks = create_chunks(n_frames, chunk_size, overlap_size)    video_chunks = []    for start, end, chunk_idx in chunks:        chunk_frames = load_video_frames(video_path, start, end)        video_chunks.append(chunk_frames)    return video_chunksdef process_inference_chunk(chunk_frames, detector):    detector.reset()    chunk_landmarks = []    for frame in chunk_frames:        lms = detector.detect_landmarks(frame)        if lms is not None:            chunk_landmarks.append(lms)        elif chunk_landmarks:            chunk_landmarks.append(chunk_landmarks[-1].copy())        else:            raise RuntimeError('No landmarks in first frame')    chunk_landmarks = np.array(chunk_landmarks)    roi_data = {}    for roi_name, roi_indices in ROIS.items():        roi_frames = []        for frame_idx, frame in enumerate(chunk_frames):            landmarks = chunk_landmarks[frame_idx]            bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)            roi_region = extract_roi_region(frame, bbox)            roi_frames.append(roi_region)        roi_data[roi_name] = np.array(roi_frames)    return {'roi_data': roi_data, 'landmarks': chunk_landmarks}print('Inference functions ready')

## 9. Summary and Next Steps

In [ ]:
print('=' * 80)print('SUMMARY')print('=' * 80)print()print('COMPLETED:')print('  1. Video frame analysis and PPG sync verification')print('  2. MediaPipe landmark detection with temporal smoothing')print('  3. Chunking strategy (450 frames, 150 overlap)')print('  4. ROI extraction with fixed 24x24 bounding boxes')print('  5. Complete preprocessing pipeline')print('  6. Showcase visualization')print('  7. Inference functions for real videos')print()print('KEY FEATURES:')print(f'  - Chunk size: {CHUNK_SIZE} frames')print(f'  - Overlap: {OVERLAP_SIZE} frames')print(f'  - ROIs: {list(ROIS.keys())}')print(f'  - ROI box size: {ROI_BOX_SIZE}')print(f'  - Temporal smoothing window: {SMOOTHING_WINDOW}')print()print('DATA ALIGNMENT:')print('  - Frame N corresponds to PPG sync file row N')print('  - Each chunk maintains exact frame-PPG alignment')print('  - Overlapping frames ensure continuity')print()print('NEXT STEPS:')print('  1. Run full dataset processing')print('  2. Use saved NPZ files for training models')print('  3. Use inference functions for real videos')print()print('Notebook complete!')